# 04 · A free LLM on my own laptop (Ollama)

Between "tiny ML model" and "big cloud LLM" there's a middle option: a **small LLM running on my laptop**.
- Free (no per-token cost)
- Private (data never leaves my computer)
- Works offline
- But less smart than the big cloud models

**Setup (once, in the terminal):**
```
brew install ollama
ollama serve              # starts Ollama (leave it running)
ollama pull llama3.2:3b   # downloads the model, about 2 GB
```

In [ ]:
import sys
sys.path.append("..")   # so Python can find the budget_agent folder

import pandas as pd
pd.set_option("display.max_colwidth", 80)
import requests

OLLAMA = "http://localhost:11434"
requests.get(f"{OLLAMA}/api/version").json()

## Step 1: Which models do I have?

In [ ]:
for m in requests.get(f"{OLLAMA}/api/tags").json()["models"]:
    print(m["name"], f"{m['size'] / 1e9:.1f} GB")

## Step 2: Why is it only 2 GB? (quantization)

The model has about 3 billion numbers ("parameters").
Stored normally (16 bits each) that's about 6 GB. Ollama stores each number in about **4 bits** instead → about 2 GB.
Slightly less accurate, but it fits in a laptop's memory and runs much faster. This trick is called **quantization**.

In [ ]:
info = requests.post(f"{OLLAMA}/api/show", json={"model": "llama3.2:3b"}).json()
info["details"]

## Step 3: Ask it something

In [ ]:
import time

def ask_local(question, system="You are a helpful assistant. Answer in 1-2 sentences."):
    start = time.perf_counter()
    reply = requests.post(f"{OLLAMA}/api/chat", json={
        "model": "llama3.2:3b",
        "messages": [{"role": "system", "content": system}, {"role": "user", "content": question}],
        "stream": False,
    }).json()
    seconds = time.perf_counter() - start
    words_per_sec = reply["eval_count"] / (reply["eval_duration"] / 1e9)
    print(reply["message"]["content"])
    print(f"\n⏱ {seconds:.2f}s   ⚡ {words_per_sec:.0f} tokens/second   💲 $0")

ask_local("What is an AI agent?")

The first call is slower because the model is loading into memory. Let me run it again:

In [ ]:
ask_local("What is an AI agent?")

## Step 4: Can it answer NovaMart questions?

I give it our policy and ask simple questions.

In [ ]:
from budget_agent.data import load_policy

shop_system = "You are NovaMart's support assistant. Answer in 1-3 short sentences using ONLY this policy:\n\n" + load_policy()

ask_local("can i pay with cash on delivery?", shop_system)

In [ ]:
ask_local("kitne din me delivery hoti hai", shop_system)

In [ ]:
ask_local("how many days do i have to return something", shop_system)

## Step 5: What it should NOT do

A small model can sound confident and still be wrong. And it has no tools: it can't look up orders or start refunds.
So in the router, the local LLM only answers **simple information questions**. Anything that changes an order goes to the big cloud LLM.

In [ ]:
ask_local("refund my order NM10033 now", shop_system)   # it can't really do this!

## What I learned
- A 3B model runs on a laptop thanks to **quantization**, for $0.
- Good for simple policy questions, even Hinglish.
- Not trusted for actions. That's the big LLM's job.

Next: put all three together → `05_the_router.ipynb`